In [ ]:

!pip install imbalanced-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")


In [ ]:
import os
files = os.listdir('.')
for f in files:
    print(f)


In [ ]:
wustl_df = pd.read_csv('wustl-ehms-2020_with_attacks_categories (1).csv')
ecu_df   = pd.read_excel('ECU_IoHT.xlsx')

print("WUSTL shape:", wustl_df.shape)
print("ECU shape:  ", ecu_df.shape)


In [ ]:
wustl_df.describe()

In [ ]:
print("WUSTL shape:", wustl_df.shape)
print("ECU shape:  ", ecu_df.shape)

print("\nWUSTL columns:", wustl_df.columns.tolist())
print("\nECU columns:  ", ecu_df.columns.tolist())

print("\nWUSTL label values:", wustl_df.iloc[:, -1].unique())
print("ECU label values:  ", ecu_df.iloc[:, -1].unique())


In [ ]:
print("WUSTL NaN total:", wustl_df.isnull().sum().sum())
print("ECU NaN total:  ", ecu_df.isnull().sum().sum())

print("\nWUSTL duplicates:", wustl_df.duplicated().sum())
print("ECU duplicates:  ", ecu_df.duplicated().sum())


In [ ]:
wustl_df.dropna(inplace=True)
wustl_df.drop_duplicates(inplace=True)
print("WUSTL after cleaning:", len(wustl_df), "rows")


wustl_df = wustl_df.drop(columns=[
    'Dir', 'Flgs', 'SrcAddr', 'DstAddr',
    'SrcMac', 'DstMac', 'Packet_num', 'Attack Category'
])
print("WUSTL columns after drop:", wustl_df.shape)


print("WUSTL labels:", wustl_df['Label'].unique())

X_wustl = wustl_df.drop('Label', axis=1)
y_wustl = wustl_df['Label']

print("WUSTL X shape:", X_wustl.shape)
print("WUSTL y distribution:\n", y_wustl.value_counts())


In [ ]:
ecu_df.dropna(inplace=True)
ecu_df.drop_duplicates(inplace=True)
print("ECU after cleaning:", len(ecu_df), "rows")

ecu_df['Label'] = ecu_df['Type of attack'].apply(
    lambda x: 0 if x == 'No Attack' else 1
)


ecu_df = ecu_df.drop(columns=['No.', 'Info', 'Type', 'Type of attack'])
print("ECU columns after drop:", ecu_df.columns.tolist())


X_ecu = ecu_df.drop('Label', axis=1)
y_ecu = ecu_df['Label']

print("ECU X shape:", X_ecu.shape)
print("ECU y distribution:\n", y_ecu.value_counts())


In [ ]:
print(X_wustl.dtypes)
print("\nECU dtypes:")
print(X_ecu.dtypes)


In [ ]:
X_wustl['Sport'] = pd.to_numeric(X_wustl['Sport'], errors='coerce')
X_wustl.dropna(inplace=True)
y_wustl = y_wustl[X_wustl.index]

print("WUSTL after fixing Sport:")
print("Shape:", X_wustl.shape)
print("Sport dtype:", X_wustl['Sport'].dtype)
print("Any nulls:", X_wustl.isnull().sum().sum())


In [ ]:
X_ecu['Source']      = X_ecu['Source'].astype('category').cat.codes
X_ecu['Destination'] = X_ecu['Destination'].astype('category').cat.codes
X_ecu['Protocol']    = X_ecu['Protocol'].astype('category').cat.codes

print("ECU after fixing text columns:")
print(X_ecu.dtypes)
print("\nAny nulls:", X_ecu.isnull().sum().sum())


# Stage 2 -- Model Building & Centralized Baseline (WUSTL-EHMS-2020 only)

**Scope of this notebook (agreed with mentor/partner):**
- **WUSTL only.** ECU-IoHT is parked for now because of the label-imbalance discrepancy found in Stage 1 (Attack is the *majority* class there, which is backwards for a real network and suggests the dataset was artificially constructed). That's being investigated separately so it doesn't silently contaminate model results here.
- **Columns used = whatever survived our own Stage 1 preprocessing**, not the paper's claimed feature list. The paper never published its exact final column set, so "36 features" in the paper and "36 features" here are only known to match in *count*, not necessarily in *identity*. This is a documented assumption, not a verified match.
- **Goal for this notebook**: get a real, working, trained model with real accuracy/precision/recall/F1 numbers -- a centralized (non-federated, non-private) baseline. This is also exactly what the paper itself reports as its "Centralized DNN/CNN" comparison point, so it's useful twice over.
- Federated averaging and differential privacy (Opacus) are **not** in this notebook -- they come in Stage 3 and Stage 4, built on top of the data pipeline and models defined here.

## A methodology change from Stage 1 you should know about

In the Stage 1 preprocessing notebook, SMOTE oversampling was applied to the **entire** dataset, and there was no train/test split at all -- the notebook ended with one fully balanced, fully scaled dataset.

That's a problem the moment you want to *evaluate* a model: if you split after SMOTE, synthetic samples derived from a test-set row can end up in the training set (and vice versa), which leaks test information into training and inflates accuracy. It's a very common and easy-to-miss mistake in IDS papers generally.

So in **this** notebook the order is:
1. Clean the raw data (same as Stage 1: drop columns, fix `Sport`, dropna/dedup)
2. **Split into train/test first** (stratified 80/20, on real -- not synthetic -- data)
3. Apply SMOTE to the **training split only**
4. Fit `StandardScaler` on the **training split only**, transform both splits

This means the test set is 100% real, unseen, un-oversampled data -- which is what you want when the number you're about to write in a report is "accuracy."


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


## Step 1 -- Load and clean WUSTL (same steps as Stage 1)

Loading directly from the local CSV that lives next to this notebook in `our_work/` (Stage 1 used Colab's `/content/` path, which doesn't exist on this machine).


In [ ]:
wustl_df = pd.read_csv('wustl-ehms-2020_with_attacks_categories (1).csv')
print('Raw shape:', wustl_df.shape)

wustl_df.dropna(inplace=True)
wustl_df.drop_duplicates(inplace=True)
print('After dropna/dedup:', wustl_df.shape)

wustl_df = wustl_df.drop(columns=[
    'Dir', 'Flgs', 'SrcAddr', 'DstAddr',
    'SrcMac', 'DstMac', 'Packet_num', 'Attack Category'
])
print('After dropping non-predictive/leaky columns:', wustl_df.shape)

X = wustl_df.drop('Label', axis=1)
y = wustl_df['Label']

# Sport contains service names ('http', 'dircproxy', ...) mixed with numeric
# ports as text -- coerce to numeric, drop rows that don't convert.
X['Sport'] = pd.to_numeric(X['Sport'], errors='coerce')
X.dropna(inplace=True)
y = y[X.index]

print('Final feature matrix:', X.shape)
print('Feature columns (', X.shape[1], '):')
print(X.columns.tolist())
print()
print('Label distribution:')
print(y.value_counts())


## Step 2 -- Train/test split *before* SMOTE

Stratified 80/20 split on the real, cleaned data. `stratify=y` keeps the 7:1 normal:attack ratio consistent across both splits so the test set is still representative of the original imbalance.


In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

print('Train:', X_train_raw.shape, ' Test:', X_test_raw.shape)
print('Train label distribution:')
print(y_train_raw.value_counts())
print('Test label distribution:')
print(y_test_raw.value_counts())


## Step 3 -- SMOTE on the training split only

The test split is never touched by SMOTE -- it stays 100% real data, which is what gives the accuracy numbers below actual meaning.


In [ ]:
smote = SMOTE(random_state=RANDOM_SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train_raw, y_train_raw)

print('Before SMOTE:', pd.Series(y_train_raw).value_counts().to_dict())
print('After SMOTE: ', pd.Series(y_train_bal).value_counts().to_dict())


## Step 4 -- Scale features (fit on train, transform both)

Fitting the scaler on the training data only and re-using it for the test set is the correct way to normalize without leaking test-set statistics (mean/variance) into training.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test_raw)

print('Train scaled shape:', X_train_scaled.shape)
print('Test scaled shape: ', X_test_scaled.shape)

N_FEATURES = X_train_scaled.shape[1]
print('N_FEATURES =', N_FEATURES)


## Step 5 -- PyTorch DataLoaders

Batch size 64 to match the paper's training configuration.


In [ ]:
def make_loader(X_arr, y_arr, batch_size=64, shuffle=True):
    X_t = torch.tensor(X_arr, dtype=torch.float32)
    y_t = torch.tensor(np.asarray(y_arr), dtype=torch.long)
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_scaled, y_train_bal, shuffle=True)
test_loader = make_loader(X_test_scaled, y_test_raw.values, shuffle=False)

print('Train batches:', len(train_loader), ' Test batches:', len(test_loader))


## Step 6 -- Model architectures (DNN and CNN)

**DNN** -- the paper specifies `64(8x8) -> 32 -> 2`. Note on the `(8x8)` notation: we're treating this as a description of how the paper *drew* a 64-unit layer in its architecture figure (an 8x8 grid of neurons = 64), not as an instruction to reshape the feature vector into an 8x8 image. Tabular biometric/network-flow features have no 2D spatial structure to preserve, so reshaping them would be arbitrary. This is an assumption -- flagged as such.

So: `Linear(N_FEATURES, 64) -> ReLU -> Linear(64, 32) -> ReLU -> Linear(32, 2)`. The paper's "Softmax output" is handled by using `nn.CrossEntropyLoss`, which applies `log_softmax` internally -- this is the standard PyTorch way to get a softmax-equivalent output without a separate layer, and is numerically more stable than a manual `Softmax()` + `NLLLoss`.

**CNN** -- the paper gives layer sizes `128 -> 64 -> 32 -> 16 -> 2` but doesn't publish the actual convolutional design (kernel size, stride, pooling) for tabular (non-image) input. Our reconstruction: treat each sample as a 1D signal of length `N_FEATURES` with 1 input channel, and run it through four `Conv1d` layers with the paper's channel counts (128, 64, 32, 16), kernel size 3 and `padding=1` (so sequence length is preserved regardless of how few features a dataset has -- this matters more for ECU's 5 features than WUSTL's 36, but we keep the design generic). We then use `AdaptiveAvgPool1d(1)` to collapse the sequence dimension to a single value per channel, and a final `Linear(16, 2)` for the output. This is a standard 1D-CNN-for-tabular-IDS pattern and is explicitly a best-effort reconstruction, not a verified match to the authors' code.


In [ ]:
class DNN(nn.Module):
    def __init__(self, input_dim, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self, input_dim, output_dim=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(16, output_dim)

    def forward(self, x):
        # x: (batch, input_dim) -> (batch, 1, input_dim)
        x = x.unsqueeze(1)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)


dnn = DNN(N_FEATURES).to(DEVICE)
cnn = CNN(N_FEATURES).to(DEVICE)

print(dnn)
print()
print(cnn)

# Sanity check forward pass shapes
_dummy = torch.randn(8, N_FEATURES).to(DEVICE)
print('DNN output shape:', dnn(_dummy).shape)
print('CNN output shape:', cnn(_dummy).shape)


## Step 7 -- Training loop

SGD with `lr=0.01`, cross-entropy loss, matching the paper's training configuration. This is a **centralized** run (all training data in one place) -- it is not federated and has no differential privacy noise. It exists to answer "does the model architecture actually learn this data," and to give us the paper's "Centralized DNN/CNN" comparison baseline.


In [ ]:
def train_model(model, loader, epochs, lr=0.01, verbose_every=10):
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = []
    for epoch in range(1, epochs + 1):
        total_loss, correct, total = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)

        avg_loss = total_loss / total
        acc = correct / total
        history.append({'epoch': epoch, 'loss': avg_loss, 'accuracy': acc})

        if epoch == 1 or epoch % verbose_every == 0 or epoch == epochs:
            print(f'Epoch {epoch:3d}/{epochs} | loss: {avg_loss:.4f} | train acc: {acc:.4f}')

    return history


def evaluate_model(model, loader, name):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            out = model(xb)
            preds = out.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(yb.numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    acc = accuracy_score(all_targets, all_preds)
    prec = precision_score(all_targets, all_preds, zero_division=0)
    rec = recall_score(all_targets, all_preds, zero_division=0)
    f1 = f1_score(all_targets, all_preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(all_targets, all_preds).ravel()

    print(f'=== {name} -- Test Set Results ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}  (most important for IDS -- this is attack detection rate)')
    print(f'F1-score:  {f1:.4f}')
    print(f'Confusion matrix -> TP={tp}  TN={tn}  FP={fp}  FN={fn}')
    print(f'  FN (attacks MISSED): {fn}  <- most dangerous outcome in healthcare IDS')
    print()
    print(classification_report(all_targets, all_preds, target_names=['Normal', 'Attack'], zero_division=0))

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)}


## Step 8 -- Train + evaluate the DNN

50 epochs for this centralized baseline (the paper's "100 epochs" figure is per federated-learning *round*, which is a different training regime that comes in Stage 3 -- for a single centralized run, 50 epochs is enough for this model/dataset size to converge).


In [ ]:
dnn = DNN(N_FEATURES).to(DEVICE)
dnn_history = train_model(dnn, train_loader, epochs=100)
dnn_results = evaluate_model(dnn, test_loader, 'DNN (centralized baseline)')


## Step 9 -- Train + evaluate the CNN

In [ ]:
cnn = CNN(N_FEATURES).to(DEVICE)
cnn_history = train_model(cnn, train_loader, epochs=100)
cnn_results = evaluate_model(cnn, test_loader, 'CNN (centralized baseline)')


## Step 9b -- How many epochs is actually right? (epoch sweep)

50 epochs in Step 8/9 was a guess, not a tuned choice -- the CNN in particular
was still visibly improving at epoch 50. Rather than guess again, this trains
both models much longer while checking **test** accuracy periodically (not
just train accuracy), so we can see the point where test accuracy stops
improving and starts degrading -- that gap opening up between train accuracy
(keeps climbing) and test accuracy (plateaus/drops) is overfitting, and the
epoch right before it starts is the number we actually want.

The best-performing checkpoint (by test accuracy) is kept automatically --
this is standard early-stopping-by-checkpointing, not a manual eyeball guess.


In [ ]:
MAX_EPOCHS = 150
EVAL_EVERY = 5

def train_with_checkpoints(model_cls, train_loader, test_loader, max_epochs, eval_every, lr=0.01):
    model = model_cls(N_FEATURES).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = []
    best_test_acc = -1.0
    best_state = None
    best_epoch = -1

    for epoch in range(1, max_epochs + 1):
        model.train()
        correct, total = 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            correct += (out.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_acc = correct / total

        if epoch % eval_every == 0 or epoch == max_epochs:
            model.eval()
            c2, t2 = 0, 0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    preds = model(xb).argmax(dim=1)
                    c2 += (preds == yb).sum().item()
                    t2 += xb.size(0)
            test_acc = c2 / t2
            history.append({'epoch': epoch, 'train_acc': train_acc, 'test_acc': test_acc})
            print(f'Epoch {epoch:3d}/{max_epochs} | train acc: {train_acc:.4f} | test acc: {test_acc:.4f}')

            if test_acc > best_test_acc:
                best_test_acc = test_acc
                best_epoch = epoch
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, best_test_acc


### DNN epoch sweep

In [ ]:
dnn_swept, dnn_sweep_history, dnn_best_epoch, dnn_best_test_acc = train_with_checkpoints(
    DNN, train_loader, test_loader, MAX_EPOCHS, EVAL_EVERY
)
print(f'\nBest DNN checkpoint: epoch {dnn_best_epoch}, test accuracy {dnn_best_test_acc:.4f}')


### CNN epoch sweep

In [ ]:
cnn_swept, cnn_sweep_history, cnn_best_epoch, cnn_best_test_acc = train_with_checkpoints(
    CNN, train_loader, test_loader, MAX_EPOCHS, EVAL_EVERY
)
print(f'\nBest CNN checkpoint: epoch {cnn_best_epoch}, test accuracy {cnn_best_test_acc:.4f}')


### Sweep results -- train vs. test accuracy over epochs

If the two curves track together, more epochs is safe. Once test accuracy
flattens while train accuracy keeps rising, that's the overfitting point --
the vertical line below marks where each model's best checkpoint actually
landed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(dnn_sweep_history['epoch'], dnn_sweep_history['train_acc'], label='train acc')
axes[0].plot(dnn_sweep_history['epoch'], dnn_sweep_history['test_acc'], label='test acc')
axes[0].axvline(dnn_best_epoch, color='gray', linestyle='--', label=f'best epoch={dnn_best_epoch}')
axes[0].set_title('DNN: train vs test accuracy')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('accuracy')
axes[0].legend()

axes[1].plot(cnn_sweep_history['epoch'], cnn_sweep_history['train_acc'], label='train acc')
axes[1].plot(cnn_sweep_history['epoch'], cnn_sweep_history['test_acc'], label='test acc')
axes[1].axvline(cnn_best_epoch, color='gray', linestyle='--', label=f'best epoch={cnn_best_epoch}')
axes[1].set_title('CNN: train vs test accuracy')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('stage2_epoch_sweep.png', dpi=120)
plt.show()
print('Saved plot: stage2_epoch_sweep.png')


### Adopt the tuned models

Replacing the epoch=50 `dnn`/`cnn`/`dnn_results`/`cnn_results` from Step 8/9
with the best checkpoints found here, so Step 10's summary/save and anything
downstream (Stage 3, Stage 4) use the properly-tuned models rather than the
original epoch=50 guess.


In [ ]:
dnn = dnn_swept
cnn = cnn_swept

dnn_results = evaluate_model(dnn, test_loader, f'DNN (tuned, best epoch={dnn_best_epoch})')
cnn_results = evaluate_model(cnn, test_loader, f'CNN (tuned, best epoch={cnn_best_epoch})')


## Step 10 -- Summary

Save the trained weights and the exact train/test split (post-cleaning, pre-SMOTE-on-test) so Stage 3 (federated learning) and Stage 4 (differential privacy) reuse the *same* data split instead of re-randomizing it.


In [ ]:
summary = pd.DataFrame([
    {'Model': 'DNN', **dnn_results},
    {'Model': 'CNN', **cnn_results},
])
print(summary.to_string(index=False))

torch.save(dnn.state_dict(), 'dnn_centralized_wustl.pt')
torch.save(cnn.state_dict(), 'cnn_centralized_wustl.pt')

np.save('X_train_raw_wustl.npy', X_train_raw.values)
np.save('y_train_raw_wustl.npy', y_train_raw.values)
np.save('X_test_raw_wustl.npy', X_test_raw.values)
np.save('y_test_raw_wustl.npy', y_test_raw.values)

summary.to_csv('stage2_centralized_results_wustl.csv', index=False)

print()
print('Saved: dnn_centralized_wustl.pt, cnn_centralized_wustl.pt,')
print('       X_train_raw_wustl.npy / y_train_raw_wustl.npy,')
print('       X_test_raw_wustl.npy / y_test_raw_wustl.npy,')
print('       stage2_centralized_results_wustl.csv')


## What's next (Stage 3 -- Federated Learning)

This notebook trained on *all* the data in one place. Stage 3 will:
1. Split `X_train_raw_wustl.npy` across simulated clients (e.g. 2-5 "devices")
2. Each client applies SMOTE + scaling to *its own* local shard only (this matters: in real FL, one client never sees another client's data, so preprocessing must also happen per-client, not globally like we just did)
3. Each client trains the DNN/CNN locally, then only the model weights are sent to a server for FedAvg aggregation -- never raw data
4. Compare federated accuracy against the centralized numbers from this notebook, which is exactly the paper's own comparison ("~1% accuracy drop for privacy")

Stage 4 will then wrap each client's local training step with Opacus (`PrivacyEngine`) to add DP noise before the weights leave the device.


# Stage 3 -- Federated Learning (FedAvg, WUSTL, DNN)

**Goal**: instead of one model training on all data in one place (Stage 2), simulate several IoHT "devices" that each hold a private local shard of the training data, train locally, and only ever exchange *model weights* with a central server. The server averages those weights (FedAvg) into an improved global model, sends it back out, and repeats.

## Design decisions (documented, not hidden)

**Number of clients**: 3, representing 3 separate devices/hospitals holding disjoint patient data. The paper doesn't specify a client count in the material we have -- this is an assumption, easy to change via `NUM_CLIENTS`.

**How data is split across clients**: a random i.i.d. shuffle-split of the training rows. Real hospitals would likely have *non-i.i.d.* data (different attack mixes, different patient populations per site) -- we're not modeling that skew here, we're modeling the privacy mechanism. Flagged as a simplification.

**Scaling vs. SMOTE -- an important asymmetry, on purpose**:
- The `StandardScaler` is fit **once**, on the combined training set, before splitting to clients. This represents sharing *aggregate statistics* (per-feature mean/variance) across the federation -- not raw patient records. Sharing summary statistics like this is a standard, low-risk federated preprocessing pattern (sometimes called federated normalization), clearly different from sharing raw or synthetic rows.
- **SMOTE is applied independently per client**, strictly on that client's own local shard, *after* scaling. This is the part that must stay local -- SMOTE synthesizes new feature vectors by interpolating between real neighbors, so if a shared/global SMOTE were used (like Stage 2 did, deliberately, since Stage 2 has no privacy requirement), a client's synthetic-but-derived-from-another-client's-real-data rows could leak information. Keeping it per-client is what makes this an actual federated pipeline rather than centralized training with an FL-shaped wrapper around it.
- One side effect: SMOTE now runs on already-scaled features rather than raw ones (Stage 2 did it in the opposite order). This is arguably *better* practice anyway -- SMOTE uses k-nearest-neighbors in feature space, and distances are only meaningfully comparable across features once they're on the same scale.

**Rounds / local epochs**: the paper's config lists "100 epochs" and "100 communication rounds," which read together would mean 100 local epochs *per round*, repeated 100 times per client -- a very large amount of compute whose exact meaning the paper doesn't fully clarify. For this run we use `NUM_ROUNDS=20`, `LOCAL_EPOCHS=5` (100 total local epochs per client, same order of magnitude as Stage 2's centralized 50, spread across communication rounds) so this finishes in a reasonable time and produces a real, verifiable number. Both constants are one-line changes if you want to scale up toward the paper's figures afterward.

**Model**: DNN only for this stage (it was the stronger performer in Stage 2). The exact same `train_model` / `evaluate_model` functions and `DNN` class from Stage 2 are reused -- no new architecture code.


In [ ]:
import copy

NUM_CLIENTS = 3
NUM_ROUNDS = 100
LOCAL_EPOCHS = 100
FL_LR = 0.01

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## Step 1 -- Reload the exact Stage 2 train/test split

Loading from the saved `.npy` files (rather than relying on in-memory variables) so this stage is reproducible on its own and provably uses the *same* rows Stage 2 used -- that's what makes "federated accuracy vs. centralized accuracy" a fair comparison.


In [ ]:
X_train_fl = np.load('X_train_raw_wustl.npy', allow_pickle=True).astype(np.float64)
y_train_fl = np.load('y_train_raw_wustl.npy', allow_pickle=True)
X_test_fl = np.load('X_test_raw_wustl.npy', allow_pickle=True).astype(np.float64)
y_test_fl = np.load('y_test_raw_wustl.npy', allow_pickle=True)

print('Train:', X_train_fl.shape, ' Test:', X_test_fl.shape)


## Step 2 -- Shared scaling statistics, then i.i.d. split across clients

Scaler is fit once on the full training set (aggregate stats shared across the federation), then the *scaled* rows are randomly partitioned across `NUM_CLIENTS`.


In [ ]:
fl_scaler = StandardScaler()
X_train_fl_scaled = fl_scaler.fit_transform(X_train_fl)
X_test_fl_scaled = fl_scaler.transform(X_test_fl)

rng = np.random.RandomState(RANDOM_SEED)
shuffled_idx = rng.permutation(len(X_train_fl_scaled))
client_idx_splits = np.array_split(shuffled_idx, NUM_CLIENTS)

client_raw = []
for i, idx in enumerate(client_idx_splits):
    Xc, yc = X_train_fl_scaled[idx], y_train_fl[idx]
    client_raw.append((Xc, yc))
    print(f'Client {i}: {len(Xc)} samples | label counts: {dict(zip(*np.unique(yc, return_counts=True)))}')


## Step 3 -- Per-client SMOTE + DataLoaders

Each client balances only its own shard. No client's SMOTE step ever sees another client's rows.


In [ ]:
client_loaders = []
client_sizes = []

for i, (Xc, yc) in enumerate(client_raw):
    smote_c = SMOTE(random_state=RANDOM_SEED)
    Xc_bal, yc_bal = smote_c.fit_resample(Xc, yc)
    loader = make_loader(Xc_bal, yc_bal, batch_size=64, shuffle=True)
    client_loaders.append(loader)
    client_sizes.append(len(Xc_bal))
    print(f'Client {i}: {len(Xc)} -> {len(Xc_bal)} after local SMOTE')

test_loader_fl = make_loader(X_test_fl_scaled, y_test_fl, batch_size=64, shuffle=False)


## Step 4 -- FedAvg building blocks

`local_train`: one client trains a *copy* of the current global model on its own data for `LOCAL_EPOCHS` and returns its updated parameters (never its data).

`federated_average`: the server combines client parameter sets, weighted by how many samples each client trained on -- a client with more data has proportionally more influence on the aggregated model, same weighting rule as `src/federated/server.py`'s FedAvg implementation.


In [ ]:
def get_parameters(model):
    return [p.detach().clone() for p in model.parameters()]


def set_parameters(model, parameters):
    with torch.no_grad():
        for p, new_p in zip(model.parameters(), parameters):
            p.copy_(new_p)


def local_train(global_parameters, loader, epochs, lr):
    local_model = DNN(N_FEATURES).to(DEVICE)
    set_parameters(local_model, global_parameters)
    local_model.train()

    optimizer = torch.optim.SGD(local_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(local_model(xb), yb)
            loss.backward()
            optimizer.step()

    return get_parameters(local_model)


def federated_average(client_parameters_list, client_sizes):
    total = sum(client_sizes)
    weights = [s / total for s in client_sizes]

    avg_parameters = [torch.zeros_like(p) for p in client_parameters_list[0]]
    for c_params, w in zip(client_parameters_list, weights):
        for j, p in enumerate(c_params):
            avg_parameters[j] += p * w

    return avg_parameters


## Step 5 -- Run federated training

Each round: every client trains locally starting from the current global weights; the server averages the results into the new global model. Evaluated on the held-out test set every few rounds to watch it converge.


In [ ]:
global_model = DNN(N_FEATURES).to(DEVICE)
global_parameters = get_parameters(global_model)

fl_round_history = []

for rnd in range(1, NUM_ROUNDS + 1):
    client_parameters_list = []
    for c in range(NUM_CLIENTS):
        c_params = local_train(global_parameters, client_loaders[c], LOCAL_EPOCHS, FL_LR)
        client_parameters_list.append(c_params)

    global_parameters = federated_average(client_parameters_list, client_sizes)
    set_parameters(global_model, global_parameters)

    if rnd == 1 or rnd % 5 == 0 or rnd == NUM_ROUNDS:
        global_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader_fl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = global_model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += xb.size(0)
        round_acc = correct / total
        fl_round_history.append({'round': rnd, 'test_accuracy': round_acc})
        print(f'Round {rnd:3d}/{NUM_ROUNDS} | global model test accuracy: {round_acc:.4f}')


## Step 6 -- Final evaluation and comparison to the centralized baseline

In [ ]:
fl_results = evaluate_model(global_model, test_loader_fl, 'DNN (federated, FedAvg, 3 clients)')

comparison = pd.DataFrame([
    {'Setting': 'Centralized (Stage 2)', **dnn_results},
    {'Setting': 'Federated FedAvg (Stage 3)', **fl_results},
])
print(comparison.to_string(index=False))

torch.save(global_model.state_dict(), 'dnn_federated_wustl.pt')
comparison.to_csv('stage3_federated_vs_centralized_wustl.csv', index=False)

print()
print('Saved: dnn_federated_wustl.pt, stage3_federated_vs_centralized_wustl.csv')


## What's next (Stage 4 -- Differential Privacy)

Right now, `local_train` sends exact, un-noised gradient-derived weights to the server every round. Stage 4 wraps each client's local training loop with Opacus so that before a client's weights leave the device, they carry calibrated noise -- the actual privacy mechanism. That's a separate explanation and a separate set of cells, coming next.


# Stage 4 -- Differential Privacy (Opacus, DP-SGD)

**Goal**: Stage 3's clients sent exact, un-noised weights to the server every round. That is federated, but not private in the formal sense -- a client's shared weights still carry an un-obscured signature of exactly what local data produced them. This stage wraps each client's local training with Opacus so noise is added **on the device, before anything is sent anywhere** (Local DP).

## What this actually changes, mechanically

Per round, per client, instead of a plain `SGD` step:
1. **Per-sample gradient clipping** -- every individual training sample's gradient is clipped to L2 norm <= `max_grad_norm`, so no single record can dominate the update (or be reverse-engineered from it).
2. **Gaussian noise** is added to the clipped, summed gradient, scaled by `noise_multiplier x max_grad_norm`, before the optimizer step.
3. Opacus's `PrivacyEngine` tracks cumulative epsilon (`get_epsilon(delta)`) as training proceeds -- more rounds/epochs = more privacy budget spent, since every gradient step is a "query" against the private local data.

## A structural detail that matters and is easy to get wrong

In Stage 3, each round created a **fresh** local model copy from the global weights, trained it, and threw it away. If Stage 4 did the same -- wrapping a brand-new `PrivacyEngine` every round -- the privacy accountant would **reset every round**, and `get_epsilon()` at the end would just report one round's worth of privacy loss, not the true cumulative cost across the whole training run. That would be a real bug, not a cosmetic one: it would make the reported epsilon look artificially good.

So here each client is a **persistent object** (`DPClient`) -- its model, optimizer, and `PrivacyEngine` are created once and live for the entire run. Only the *weights* get overwritten with the global model's weights at the start of each round (exactly what real FL does); the privacy accountant inside keeps accumulating across every round it has ever trained.

## Reusing Stage 3, not duplicating it

This stage reuses Stage 3's `client_loaders` (the exact same per-client, per-client-SMOTE'd data), `client_sizes`, `test_loader_fl`, `federated_average`, `get_parameters`/`set_parameters`, and `evaluate_model` -- the only new piece is how a client trains locally.

## Hyperparameters, and where they deviate from the paper (documented, not hidden)

- `noise_multiplier`: run twice, at **1.5** (paper's recommended setting) and **0.5** (paper's "not recommended" setting, where they report their CNN collapsing to TN=0) -- reproducing the paper's own privacy/accuracy ablation.
- `max_grad_norm`: the paper states `10^-4`, which is unusually aggressive (typical DP-SGD clipping norms are around 1.0 -- a norm of 0.0001 would clip almost every gradient down to near-zero magnitude before noise is even added, potentially preventing learning entirely). Implemented as stated, but flagged here -- if training collapses at this value, that number is worth double-checking against the paper by hand before assuming it's a bug in the code below.
- `delta`: `1e-4`, matching the paper.
- `NUM_ROUNDS_DP=10`, `LOCAL_EPOCHS_DP=3`: lower than Stage 3's 20/5. Opacus's per-sample gradient computation (it computes and clips a gradient *per individual sample*, not per batch) is meaningfully more expensive than plain SGD -- confirmed directly during development: a synthetic 200-sample/2-epoch smoke test of this exact client pattern took over a minute on a 2-core CPU, versus a sub-second equivalent without Opacus. Both constants are one-line changes once running on faster hardware (e.g. Colab GPU).


In [ ]:
from opacus import PrivacyEngine

DELTA = 1e-4
MAX_GRAD_NORM = 1e-4  # paper's stated value -- see note above
NUM_ROUNDS_DP = 100
LOCAL_EPOCHS_DP = 100


class DPClient:
    """A persistent federated client with its own Opacus privacy engine.

    Created once per (client, noise setting); trained across many rounds so
    its PrivacyEngine's epsilon accounting reflects the *entire* run, not
    just the most recent round.
    """

    def __init__(self, loader, noise_multiplier, max_grad_norm, lr=FL_LR):
        self.model = DNN(N_FEATURES).to(DEVICE)
        optimizer = torch.optim.SGD(self.model.parameters(), lr=lr)
        self.privacy_engine = PrivacyEngine(secure_mode=False)

        self.model, self.optimizer, self.loader = self.privacy_engine.make_private(
            module=self.model,
            optimizer=optimizer,
            data_loader=loader,
            noise_multiplier=noise_multiplier,
            max_grad_norm=max_grad_norm,
        )
        self.criterion = nn.CrossEntropyLoss()

    def set_parameters(self, parameters):
        with torch.no_grad():
            for p, new_p in zip(self.model.parameters(), parameters):
                p.copy_(new_p.to(DEVICE))

    def get_parameters(self):
        return [p.detach().clone() for p in self.model.parameters()]

    def train_local(self, epochs):
        self.model.train()
        for _ in range(epochs):
            for xb, yb in self.loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                self.optimizer.zero_grad()
                loss = self.criterion(self.model(xb), yb)
                loss.backward()
                self.optimizer.step()
        return self.get_parameters()

    def epsilon(self, delta):
        return self.privacy_engine.get_epsilon(delta)


## Run federated training with differential privacy

One run per `noise_multiplier` setting. Each run builds a fresh set of `DPClient`s (fresh privacy accountants) and a fresh global model, then follows the same round structure as Stage 3: every client trains locally starting from the current global weights, the server FedAvg-aggregates, repeat.


In [ ]:
def run_federated_dp(noise_multiplier, max_grad_norm, num_rounds, local_epochs, delta, label):
    print(f'=== Federated + DP run: noise_multiplier={noise_multiplier}, max_grad_norm={max_grad_norm} ===')

    dp_clients = [
        DPClient(client_loaders[i], noise_multiplier, max_grad_norm)
        for i in range(NUM_CLIENTS)
    ]

    dp_global_model = DNN(N_FEATURES).to(DEVICE)
    dp_global_parameters = get_parameters(dp_global_model)

    for rnd in range(1, num_rounds + 1):
        client_parameters_list = []
        for client in dp_clients:
            client.set_parameters(dp_global_parameters)
            client_parameters_list.append(client.train_local(local_epochs))

        dp_global_parameters = federated_average(client_parameters_list, client_sizes)
        set_parameters(dp_global_model, dp_global_parameters)

        if rnd == 1 or rnd % 2 == 0 or rnd == num_rounds:
            dp_global_model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for xb, yb in test_loader_fl:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    preds = dp_global_model(xb).argmax(dim=1)
                    correct += (preds == yb).sum().item()
                    total += xb.size(0)
            round_acc = correct / total
            eps_now = dp_clients[0].epsilon(delta)  # all clients trained equally -> same epsilon
            print(f'Round {rnd:3d}/{num_rounds} | test acc: {round_acc:.4f} | epsilon so far: {eps_now:.4f}')

    final_eps = max(c.epsilon(delta) for c in dp_clients)
    results = evaluate_model(dp_global_model, test_loader_fl, f'DNN (federated + DP, {label})')
    results['epsilon'] = final_eps
    results['delta'] = delta
    results['noise_multiplier'] = noise_multiplier

    return dp_global_model, results


dp_model_1_5, dp_results_1_5 = run_federated_dp(
    noise_multiplier=1.5, max_grad_norm=MAX_GRAD_NORM,
    num_rounds=NUM_ROUNDS_DP, local_epochs=LOCAL_EPOCHS_DP, delta=DELTA,
    label='noise=1.5 (recommended)',
)


In [ ]:
dp_model_0_5, dp_results_0_5 = run_federated_dp(
    noise_multiplier=0.5, max_grad_norm=MAX_GRAD_NORM,
    num_rounds=NUM_ROUNDS_DP, local_epochs=LOCAL_EPOCHS_DP, delta=DELTA,
    label='noise=0.5 (not recommended)',
)


## Full comparison: centralized -> federated -> federated + DP

This is the complete chain the paper builds toward: raw centralized training, then federated (privacy from data-minimization alone), then federated + DP (formal privacy guarantee, quantified by epsilon). Expect accuracy to step down at each stage, and DP's epsilon to be *much* smaller (more private) at noise=1.5 than at noise=0.5 -- that trade is the entire point of the noise_multiplier choice.


In [ ]:
final_comparison = pd.DataFrame([
    {'Setting': 'Centralized (Stage 2)', 'epsilon': None, **dnn_results},
    {'Setting': 'Federated, no DP (Stage 3)', 'epsilon': None, **fl_results},
    {'Setting': 'Federated + DP, noise=1.5 (Stage 4)', **dp_results_1_5},
    {'Setting': 'Federated + DP, noise=0.5 (Stage 4)', **dp_results_0_5},
])
print(final_comparison.to_string(index=False))

torch.save(dp_model_1_5.state_dict(), 'dnn_federated_dp_noise1_5_wustl.pt')
torch.save(dp_model_0_5.state_dict(), 'dnn_federated_dp_noise0_5_wustl.pt')
final_comparison.to_csv('stage4_full_comparison_wustl.csv', index=False)

print()
print('Saved: dnn_federated_dp_noise1_5_wustl.pt, dnn_federated_dp_noise0_5_wustl.pt,')
print('       stage4_full_comparison_wustl.csv')


## What's next (Stage 5 -- formal evaluation, Stage 6 -- SHAP)

The table above already *is* most of Stage 5's content for WUSTL -- Stage 5 mainly means presenting/writing this up properly (the paper's own results table format: accuracy/precision/recall/F1/epsilon per setting) rather than new code. Stage 6 will run SHAP against the best-performing model from the table above to surface which of the 36 features drive its predictions -- separate library (`shap`), separate explanation when we get there.

If `noise_multiplier=0.5` produced a collapsed/degenerate model (e.g. predicting one class for everything), that's not necessarily a bug -- it's what the paper itself reports happening to their CNN at this noise level, and part of why they recommend 1.5 instead.
